# 04 — Model Evaluation & Comparison

This notebook evaluates all mechanistic eGFR equations (CKD-EPI 2021, MDRD,
Cockcroft-Gault) and the hybrid ML models on an internal test set.

**Contents:**
1. Data preparation — load NHANES or synthetic data and split into train/test
2. Model training — fit Ridge, Random Forest, and LightGBM on the training set
3. Comprehensive evaluation — P30, P10, RMSE, MAE, bias, Pearson r, CKD stage agreement
4. Bland-Altman plots — agreement analysis for each estimator
5. CKD-stage reclassification analysis — confusion matrices
6. Hybrid ML vs individual equations — head-to-head comparison
7. Bootstrap confidence intervals — uncertainty quantification

---

In [ ]:
import sys, os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Ensure package is importable
sys.path.insert(0, os.path.abspath(".."))

from eGFR.models import (
    calc_egfr_ckd_epi_2021,
    calc_egfr_mdrd,
    calc_crcl_cockcroft_gault,
)
from eGFR.train import (
    create_features,
    stratified_split,
    train_ridge,
    train_random_forest,
    train_lightgbm,
)
from eGFR.evaluate import (
    evaluate_model,
    bland_altman_stats,
    p30_accuracy,
    p10_accuracy,
    bootstrap_ci,
)
from eGFR.utils import egfr_to_ckd_stage
from eGFR.data import generate_synthetic_mgfr_dataset

warnings.simplefilter("ignore", UserWarning)  # suppress MDRD warnings

print("All imports loaded successfully.")

## 1. Data Preparation

We attempt to load real NHANES XPT data from `data/raw/`. If unavailable,
we fall back to a realistic synthetic dataset.

In [ ]:
# --- Try loading real NHANES data; fall back to synthetic ----------------
data_loaded = False
try:
    from eGFR.data import read_xpt, clean_kidney_data
    raw_dir = os.path.join("..", "data", "raw")
    biopro = read_xpt(os.path.join(raw_dir, "BIOPRO_I.XPT"))
    demo   = read_xpt(os.path.join(raw_dir, "DEMO_I.XPT"))
    bmx    = read_xpt(os.path.join(raw_dir, "BMX_I.XPT"))
    df_all = clean_kidney_data(biopro, demo, bmx)
    data_loaded = True
    print(f"Loaded real NHANES data: {len(df_all)} records")
except Exception as e:
    print(f"Real data unavailable ({e}). Generating synthetic dataset...")

if not data_loaded:
    # Generate a larger synthetic dataset for robust evaluation
    df_synth = generate_synthetic_mgfr_dataset(n_samples=2000, random_state=42)
    # Compute CKD-EPI 2021 eGFR as the "reference" for test-set evaluation
    df_all = df_synth.copy()
    # Use measured_gfr as the target (ground truth)
    print(f"Generated synthetic dataset: {len(df_all)} records")

print(f"Columns: {list(df_all.columns)}")
df_all.describe().round(2)

## 2. Train/Test Split & Model Training

We split the data, engineer features, and train Ridge, Random Forest, and
LightGBM models for comparison with procedural estimators.

In [ ]:
# --- Feature engineering & split -----------------------------------------
X, feature_names = create_features(df_all)

# Target: measured_gfr if available, else CKD-EPI 2021 eGFR
if "measured_gfr" in df_all.columns:
    y = df_all["measured_gfr"].values
    target_label = "Measured GFR"
else:
    y = np.array([
        calc_egfr_ckd_epi_2021(row.cr_mgdl, row.age_years, row.sex)
        for _, row in df_all.iterrows()
    ])
    target_label = "CKD-EPI 2021 eGFR"

# Remove NaN targets
valid = ~np.isnan(y)
X_clean = X[valid]
y_clean = y[valid]

# 70/30 split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean, test_size=0.3, random_state=42
)

print(f"Target: {target_label}")
print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set:  {X_test.shape[0]} samples")
print(f"Features:  {X_train.shape[1]} ({feature_names})")

# --- Train ML models -----------------------------------------------------
ridge_model = train_ridge(X_train, y_train, alpha=1.0)
rf_model    = train_random_forest(X_train, y_train, n_estimators=200)
lgb_model   = train_lightgbm(X_train, y_train, X_test, y_test)

print("\nModels trained: Ridge, Random Forest (200 trees), LightGBM (early stopping)")

## 3. Comprehensive Evaluation — All Models & Estimators

We compute predictions for each method on the test set and evaluate with
`evaluate_model()` (RMSE, MAE, bias, Pearson r, P30, P10, Bland-Altman, CKD concordance).

In [ ]:
# --- Compute predictions on test set -------------------------------------
# Mechanistic equations need raw clinical values from test rows
test_indices = np.where(valid)[0][
    train_test_split(
        range(int(valid.sum())), test_size=0.3, random_state=42
    )[1]
]
df_test = df_all.iloc[test_indices].reset_index(drop=True)

# Mechanistic estimator predictions
pred_ckd_epi = np.array([
    calc_egfr_ckd_epi_2021(row.cr_mgdl, row.age_years, row.sex)
    for _, row in df_test.iterrows()
])

pred_mdrd = np.array([
    calc_egfr_mdrd(row.cr_mgdl, row.age_years, row.sex)
    for _, row in df_test.iterrows()
])

pred_cg = np.array([
    calc_crcl_cockcroft_gault(row.cr_mgdl, row.age_years, row.weight_kg, row.sex)
    for _, row in df_test.iterrows()
])

# ML model predictions
pred_ridge = ridge_model.predict(X_test)
pred_rf    = rf_model.predict(X_test)
pred_lgb   = lgb_model.predict(X_test)

# --- Evaluate all models -------------------------------------------------
all_predictions = {
    "CKD-EPI 2021": pred_ckd_epi,
    "MDRD": pred_mdrd,
    "Cockcroft-Gault": pred_cg,
    "Ridge": pred_ridge,
    "Random Forest": pred_rf,
    "LightGBM": pred_lgb,
}

results = []
for name, preds in all_predictions.items():
    res = evaluate_model(y_test, preds, model_name=name)
    results.append(res)

# Display results table
results_df = pd.DataFrame(results)[
    ["model_name", "rmse", "mae", "bias", "r_pearson", "p30", "p10", "ckd_stage_agreement"]
].rename(columns={
    "model_name": "Model",
    "rmse": "RMSE",
    "mae": "MAE",
    "bias": "Bias",
    "r_pearson": "Pearson r",
    "p30": "P30 (%)",
    "p10": "P10 (%)",
    "ckd_stage_agreement": "CKD Stage Agr (%)",
})

print("\n=== Model Evaluation on Internal Test Set ===")
print(f"Target: {target_label} | N = {len(y_test)}\n")
print(results_df.to_string(index=False, float_format="{:.2f}".format))

## 4. Bland-Altman Plots

Bland-Altman plots assess agreement between each estimator and the reference.
The x-axis shows the mean of true and predicted values; the y-axis shows
the difference (predicted − true). Horizontal lines mark the mean bias and
95% limits of agreement.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Bland-Altman Plots: Estimated vs Reference GFR", fontsize=16, y=0.98)

for ax, (name, preds) in zip(axes.flat, all_predictions.items()):
    means = (y_test + preds) / 2.0
    diffs = preds - y_test
    ba = bland_altman_stats(y_test, preds)

    ax.scatter(means, diffs, alpha=0.3, s=10, color="steelblue")
    ax.axhline(ba["mean_bias"], color="red", linestyle="-", lw=1.5,
               label=f'Bias = {ba["mean_bias"]:.1f}')
    ax.axhline(ba["loa_upper"], color="orange", linestyle="--", lw=1,
               label=f'+1.96 SD = {ba["loa_upper"]:.1f}')
    ax.axhline(ba["loa_lower"], color="orange", linestyle="--", lw=1,
               label=f'−1.96 SD = {ba["loa_lower"]:.1f}')
    ax.axhline(0, color="gray", linestyle=":", lw=0.8)
    ax.set_title(name, fontsize=13)
    ax.set_xlabel("Mean of True & Predicted")
    ax.set_ylabel("Predicted − True")
    ax.legend(fontsize=8, loc="upper left")

plt.tight_layout()
plt.savefig("04_bland_altman.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 04_bland_altman.png")

## 5. CKD-Stage Reclassification Analysis

We map true and predicted GFR values to CKD stages (G1–G5) and compute
confusion matrices to identify reclassification patterns.

In [ ]:
CKD_STAGES = ["G1", "G2", "G3a", "G3b", "G4", "G5"]
true_stages = pd.Series([egfr_to_ckd_stage(v) for v in y_test])

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle("CKD-Stage Reclassification: True vs Predicted", fontsize=16, y=0.98)

for ax, (name, preds) in zip(axes.flat, all_predictions.items()):
    pred_stages = pd.Series([egfr_to_ckd_stage(v) for v in preds])
    ct = pd.crosstab(true_stages, pred_stages, dropna=False)
    ct = ct.reindex(index=CKD_STAGES, columns=CKD_STAGES, fill_value=0)

    # Compute reclassification rate
    total = ct.values.sum()
    concordant = np.trace(ct.values)
    agreement = concordant / total * 100 if total > 0 else 0

    # Heatmap
    im = ax.imshow(ct.values, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(CKD_STAGES)))
    ax.set_xticklabels(CKD_STAGES, fontsize=9)
    ax.set_yticks(range(len(CKD_STAGES)))
    ax.set_yticklabels(CKD_STAGES, fontsize=9)
    ax.set_xlabel("Predicted Stage")
    ax.set_ylabel("True Stage")
    ax.set_title(f"{name}\nAgreement: {agreement:.1f}%", fontsize=12)

    # Annotate cells
    for i in range(len(CKD_STAGES)):
        for j in range(len(CKD_STAGES)):
            val = ct.values[i, j]
            if val > 0:
                color = "white" if val > ct.values.max() * 0.5 else "black"
                ax.text(j, i, str(val), ha="center", va="center",
                        fontsize=8, color=color)

plt.tight_layout()
plt.savefig("04_ckd_reclassification.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 04_ckd_reclassification.png")

## 6. Hybrid ML vs Individual Equations

Direct comparison of the best ML model (LightGBM) against each mechanistic
equation. Scatter plots show predicted vs true GFR with the identity line.

In [ ]:
# --- Identify best ML model by RMSE --------------------------------------
ml_results = {name: res for name, res in zip(all_predictions.keys(), results)
              if name in ["Ridge", "Random Forest", "LightGBM"]}
best_ml_name = min(ml_results, key=lambda k: ml_results[k]["rmse"])
best_ml_pred = all_predictions[best_ml_name]
best_ml_res  = ml_results[best_ml_name]

mechanistic = {
    "CKD-EPI 2021": (pred_ckd_epi, results[0]),
    "MDRD": (pred_mdrd, results[1]),
    "Cockcroft-Gault": (pred_cg, results[2]),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
fig.suptitle(f"Best ML ({best_ml_name}) vs Mechanistic Equations", fontsize=15, y=1.02)

for ax, (eq_name, (eq_pred, eq_res)) in zip(axes, mechanistic.items()):
    lims = [0, max(y_test.max(), eq_pred.max(), best_ml_pred.max()) * 1.05]

    ax.scatter(y_test, eq_pred, alpha=0.25, s=12, label=f"{eq_name} (RMSE={eq_res['rmse']:.1f})",
               color="coral")
    ax.scatter(y_test, best_ml_pred, alpha=0.25, s=12,
               label=f"{best_ml_name} (RMSE={best_ml_res['rmse']:.1f})", color="steelblue")
    ax.plot(lims, lims, "k--", lw=1, alpha=0.5, label="Identity")
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel(f"True ({target_label})")
    ax.set_ylabel("Predicted")
    ax.set_title(f"{best_ml_name} vs {eq_name}", fontsize=12)
    ax.legend(fontsize=8, loc="upper left")
    ax.set_aspect("equal")

plt.tight_layout()
plt.savefig("04_ml_vs_equations.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nBest ML model: {best_ml_name}")
print(f"  RMSE: {best_ml_res['rmse']:.2f}  |  P30: {best_ml_res['p30']:.1f}%  |  "
      f"Bias: {best_ml_res['bias']:.2f}  |  CKD Agreement: {best_ml_res['ckd_stage_agreement']:.1f}%")

## 7. P30 Accuracy with Bootstrap Confidence Intervals

We report P30 accuracy for each estimator with 95% bootstrap CIs to
quantify the uncertainty of the accuracy metric.

In [ ]:
print("P30 Accuracy with 95% Bootstrap CIs (n_bootstrap=2000)")
print("=" * 65)

p30_rows = []
for name, preds in all_predictions.items():
    p30_val = p30_accuracy(y_test, preds)
    lo, hi, mean = bootstrap_ci(y_test, preds, metric_func=p30_accuracy,
                                n_bootstrap=2000, ci=95.0, random_state=42)
    p30_rows.append({
        "Model": name,
        "P30 (%)": f"{p30_val:.1f}",
        "95% CI": f"[{lo:.1f}, {hi:.1f}]",
        "≥85% Target": "✓" if p30_val >= 85.0 else "✗",
    })

p30_df = pd.DataFrame(p30_rows)
print(p30_df.to_string(index=False))

## 8. Summary & Interpretation

### Key Findings

- **CKD-EPI 2021** is the current KDIGO-recommended equation and typically shows
  the lowest bias and best P30 among mechanistic estimators.
- **MDRD** tends to underestimate GFR at higher values (>60 mL/min/1.73 m²).
  This is expected and the reason it is less recommended for screening.
- **Cockcroft-Gault** returns CrCl (mL/min, not normalised to BSA), so direct
  comparison with eGFR is approximate. It remains essential for drug dosing.
- **Hybrid ML models** (especially LightGBM) combine mechanistic equation outputs
  with raw biomarkers, potentially capturing nonlinear patterns missed by
  individual equations.

### Clinical Caveats

- If using synthetic data, results demonstrate the pipeline rather than
  clinical validity. Real validation requires measured GFR (see notebook 05).
- CKD stage reclassification near thresholds (eGFR 60, 45, 30, 15) has
  high clinical impact — always report reclassification rates.
- P30 ≥ 85% is the KDIGO benchmark for clinical acceptability.